In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, getpass
os.environ['KAGGLE_USERNAME'] = "mathieubonneau2708"
os.environ['KAGGLE_KEY'] = "KGAT_1836daa6be71d1f6577f52797c2797c9"
!pip install -q kaggle
!kaggle datasets download -d divyanshusingh369/complete-pokemon-library-32k-images-and-csv
!mkdir -p /content/pokemon_data
!unzip -q complete-pokemon-library-32k-images-and-csv.zip -d /content/pokemon_data

Mounted at /content/drive
Dataset URL: https://www.kaggle.com/datasets/divyanshusingh369/complete-pokemon-library-32k-images-and-csv
License(s): apache-2.0
100% 514M/514M [00:34<00:00, 15.6MB/s]



In [ ]:

import math
import os
import random
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple, List, Dict

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageSequence
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils
from torchvision.transforms import InterpolationMode


# ============================================================
# Configuration
# ============================================================

TYPE_LIST = ["Fire", "Water", "Grass", "Electric"]
TYPE_TO_IDX = {t.lower(): i for i, t in enumerate(TYPE_LIST)}


@dataclass
class Config:
    # Data
    images_root: str = "/content/pokemon_data/Pokemon Dataset/Pokemon Dataset"
    metadata_csv: str = "/content/pokemon_data/pokemonDB_dataset.csv"
    image_size: int = 64
    limit_dataset: Optional[int] = None

    include_gif: bool = False
    shiny_mode: str = "both"      # "both", "normal", "shiny"
    path_must_contain: Optional[List[str]] = field(default_factory=lambda: ["normal"])
    path_must_not_contain: Optional[List[str]] = field(default_factory=lambda: ["back", "shiny", "back, shiny"])

    # Minimal conditional test setup
    allowed_primary_types: List[str] = field(default_factory=lambda: TYPE_LIST.copy())
    require_monotype: bool = True

    # Output / runtime
    output_dir: str = "./drive/MyDrive/outputs_pokemon_dataset_conditional_4types_64"
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 48
    batch_size: int = 128
    use_amp: bool = True

    # Stage control / checkpoint loading
    run_stage1: bool = True
    run_stage2: bool = True
    vae_ckpt_path: Optional[str] = "/content/drive/MyDrive/outputs_pokemon_dataset_conditional_from_nv/stage1_vae/vae_last.pt"
    diff_ckpt_path: Optional[str] = "/content/drive/MyDrive/outputs_pokemon_dataset_conditional_4types_64/stage2_latent_diffusion/latent_diffusion_last.pt"

    # Stage 1: VAE (kept very close to latent_diffusion_nv.ipynb)
    vae_epochs: int = 150
    vae_lr: float = 2e-5
    vae_latent_channels: int = 4
    vae_base_channels: int = 64
    vae_kl_weight: float = 1e-4
    vae_recon_mse_weight: float = 0.05
    vae_grad_clip: float = 1.0
    vae_save_every: int = 10

    # Stage 2: latent diffusion
    diff_timesteps: int = 300
    diff_train_steps: int = 500000
    diff_lr: float = 2e-5
    diff_base_channels: int = 128
    diff_objective: str = "v"    # "v" or "eps"
    diff_min_snr_gamma: float = 5.0
    diff_ema_decay: float = 0.995
    diff_ema_start: int = 500
    diff_grad_clip: float = 1.0
    diff_sample_every: int = 1000
    diff_save_every: int = 5000
    diff_ddim_steps: int = 100

    # Sampling / latent settings
    diff_use_posterior_sample: bool = True
    diff_posterior_temperature: float = 1.0
    sample_temperature: float = 1.12
    sample_eta: float = 0.20

    # Conditional settings
    cond_drop_prob: float = 0.10
    guidance_scale: float = 0.0
    sample_type_names: List[str] = field(default_factory=lambda: [
        "Fire", "Water", "Grass", "Electric",
        "Fire", "Water", "Grass", "Electric",
    ])


CFG = Config()


# ============================================================
# Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def device_autocast(device: torch.device, enabled: bool):
    if device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=enabled)
    return torch.autocast(device_type="cpu", dtype=torch.bfloat16, enabled=False)


def denorm(x: torch.Tensor) -> torch.Tensor:
    return x.clamp(-1, 1).add(1).div(2)


def save_grid(images: torch.Tensor, path: str, nrow: int = 4) -> None:
    utils.save_image(denorm(images.detach().cpu()), path, nrow=nrow)


def cycle(loader):
    while True:
        for batch in loader:
            yield batch


def num_groups(channels: int, max_groups: int = 8) -> int:
    for g in reversed(range(1, max_groups + 1)):
        if channels % g == 0:
            return g
    return 1


def normalize_name(name: str) -> str:
    s = name.strip()
    s = s.replace("’", "'").replace("`", "'")
    s = s.replace("_", " ")
    s = s.replace("♀", " female ")
    s = s.replace("♂", " male ")
    s = re.sub(r"\((.*?)\)", r" \1 ", s)
    s = re.sub(r"[^a-zA-Z0-9]+", " ", s.lower())
    s = re.sub(r"\s+", " ", s).strip()
    return s


SPECIAL_ALIASES = {
    "nidoran female": "Nidoran♀ (female) Nidoran♀",
    "nidoran male": "Nidoran♂ (male) Nidoran♂",
}


def split_type_string(type_str: str) -> List[str]:
    if not isinstance(type_str, str):
        return []
    return [p.strip() for p in type_str.split(",") if p.strip()]


def parse_type_string(type_str: str) -> torch.Tensor:
    vec = torch.zeros(len(TYPE_LIST), dtype=torch.float32)
    parts = split_type_string(type_str)
    if not parts:
        return vec
    primary = parts[0]
    idx = TYPE_TO_IDX.get(primary.lower())
    if idx is not None:
        vec[idx] = 1.0
    return vec


def cond_vector_from_types(type_names: List[str]) -> torch.Tensor:
    vec = torch.zeros(len(TYPE_LIST), dtype=torch.float32)
    for name in type_names:
        idx = TYPE_TO_IDX.get(name.lower())
        if idx is not None:
            vec[idx] = 1.0
    return vec


def cond_vector_from_string(type_string: str) -> torch.Tensor:
    parts = split_type_string(type_string)
    if not parts:
        return torch.zeros(len(TYPE_LIST), dtype=torch.float32)
    return cond_vector_from_types([parts[0]])


def make_sample_conditions(type_names: List[str], device: torch.device) -> torch.Tensor:
    conds = [cond_vector_from_types([t]) for t in type_names]
    return torch.stack(conds, dim=0).to(device)


# ============================================================
# Dataset
# ============================================================

class PokemonDatasetConditional(Dataset):
    ALLOWED_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp"}

    def __init__(
        self,
        images_root: str,
        metadata_csv: str,
        image_size: int,
        include_gif: bool = False,
        shiny_mode: str = "both",
        path_must_contain: Optional[List[str]] = None,
        path_must_not_contain: Optional[List[str]] = None,
        limit_dataset: Optional[int] = None,
    ):
        self.images_root = Path(images_root)
        self.metadata_csv = Path(metadata_csv)
        self.include_gif = include_gif
        self.shiny_mode = shiny_mode.lower()
        self.path_must_contain = [s.lower() for s in path_must_contain] if path_must_contain else None
        self.path_must_not_contain = [s.lower() for s in path_must_not_contain] if path_must_not_contain else None
        self.allowed_primary_types = {t.lower() for t in CFG.allowed_primary_types}
        self.require_monotype = CFG.require_monotype

        if self.shiny_mode not in {"both", "normal", "shiny"}:
            raise ValueError("shiny_mode must be one of: 'both', 'normal', 'shiny'")

        if not self.images_root.exists():
            raise FileNotFoundError(f"images_root not found: {self.images_root}")
        if not self.metadata_csv.exists():
            raise FileNotFoundError(f"metadata_csv not found: {self.metadata_csv}")

        self.metadata_by_norm = self._load_metadata(self.metadata_csv)
        self.samples = self._collect_samples()

        if limit_dataset is not None:
            self.samples = self.samples[:limit_dataset]

        if not self.samples:
            raise RuntimeError("No matched samples found in images_root.")

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size), interpolation=InterpolationMode.NEAREST),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

    def _load_metadata(self, csv_path: Path) -> Dict[str, Dict]:
        df = pd.read_csv(csv_path)
        required = {"Pokemon", "Type"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"metadata csv missing columns: {missing}")

        meta = {}
        for _, row in df.iterrows():
            canonical = str(row["Pokemon"]).strip()
            type_str = str(row["Type"]).strip()

            entry = {
                "canonical_name": canonical,
                "type_str": type_str,
                "cond": parse_type_string(type_str),
            }

            keys = {normalize_name(canonical)}

            # extra aliases for forms / awkward metadata rows
            if canonical in SPECIAL_ALIASES.values():
                for alias_norm, alias_name in SPECIAL_ALIASES.items():
                    if alias_name == canonical:
                        keys.add(alias_norm)

            # alias without parenthetical text
            alias_simple = re.sub(r"\([^)]*\)", " ", canonical)
            keys.add(normalize_name(alias_simple))

            for k in keys:
                if k:
                    meta[k] = entry
        return meta

    def _match_entry_from_folder(self, folder_name: str) -> Optional[Dict]:
        norm = normalize_name(folder_name)
        entry = self.metadata_by_norm.get(norm)
        if entry is not None:
            return entry
        if norm in SPECIAL_ALIASES:
            return self.metadata_by_norm.get(normalize_name(SPECIAL_ALIASES[norm]))
        return None

    def _path_is_allowed(self, path: Path) -> bool:
        suffix = path.suffix.lower()
        if suffix == ".gif" and not self.include_gif:
            return False
        if suffix != ".gif" and suffix not in self.ALLOWED_IMAGE_SUFFIXES:
            return False

        path_str = str(path).lower()

        if self.path_must_contain and not all(s in path_str for s in self.path_must_contain):
            return False
        if self.path_must_not_contain and any(s in path_str for s in self.path_must_not_contain):
            return False

        parts = [p.lower() for p in path.parts]
        has_shiny = "shiny" in parts
        has_normal = "normal" in parts

        if self.shiny_mode == "shiny" and not has_shiny:
            return False
        if self.shiny_mode == "normal" and has_shiny:
            return False

        # if user asked normal but paths don't explicitly contain "normal", keep them
        # unless they are explicitly shiny
        return True

    def _collect_samples(self) -> List[Dict]:
        samples = []
        non_matched = []

        all_files = sorted([p for p in self.images_root.rglob("*") if p.is_file()])
        for path in all_files:
            if not self._path_is_allowed(path):
                continue

            rel = path.relative_to(self.images_root)
            if len(rel.parts) < 2:
                non_matched.append(str(rel))
                continue

            pokemon_folder = rel.parts[0]
            entry = self._match_entry_from_folder(pokemon_folder)
            if entry is None:
                non_matched.append(str(rel))
                continue

            type_parts = split_type_string(entry["type_str"])
            if not type_parts:
                continue
            primary = type_parts[0].lower()
            if primary not in self.allowed_primary_types:
                continue
            if self.require_monotype and len(type_parts) != 1:
                continue

            samples.append({
                "path": path,
                "rel_path": str(rel),
                "pokemon_folder": pokemon_folder,
                "canonical_name": entry["canonical_name"],
                "type_str": entry["type_str"],
                "cond": entry["cond"],
            })

        self.non_matched_examples = non_matched[:20]
        return samples

    def __len__(self) -> int:
        return len(self.samples)

    @staticmethod
    def load_image(path: Path) -> Image.Image:
        suffix = path.suffix.lower()
        if suffix == ".gif":
            with Image.open(path) as img:
                img.seek(0)
                frame = img.convert("RGBA")
            white_bg = Image.new("RGBA", frame.size, (255, 255, 255, 255))
            composited = Image.alpha_composite(white_bg, frame)
            return composited.convert("RGB")

        img = Image.open(path)
        if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
            rgba = img.convert("RGBA")
            white_bg = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
            composited = Image.alpha_composite(white_bg, rgba)
            return composited.convert("RGB")
        return img.convert("RGB")

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        img = self.load_image(sample["path"])
        img = self.transform(img)
        cond = sample["cond"].clone()
        return img, cond


def inspect_dataset_mapping(dataset: PokemonDatasetConditional, max_examples: int = 12) -> None:
    print(f"Racine images : {dataset.images_root}")
    print(f"CSV metadata : {dataset.metadata_csv}")
    print(f"Images matchées : {len(dataset.samples)}")
    print(f"Images non matchées (exemples) : {len(getattr(dataset, 'non_matched_examples', []))}")

    covered = len({s["canonical_name"] for s in dataset.samples})
    print(f"Pokémon couverts : {covered}")
    print(f"Types autorisés : {sorted(dataset.allowed_primary_types)}")
    print(f"Monotype uniquement : {dataset.require_monotype}")

    print("\n" + "=" * 80)
    print("EXEMPLES DE MATCHES")
    print("=" * 80)
    for s in dataset.samples[:max_examples]:
        print(f'{s["rel_path"]:<80} -> {s["canonical_name"]} [{s["type_str"]}]')

    if getattr(dataset, "non_matched_examples", None):
        print("\n" + "=" * 80)
        print("EXEMPLES NON MATCHÉS")
        print("=" * 80)
        for p in dataset.non_matched_examples[:max_examples]:
            print(p)


# ============================================================
# Core modules
# ============================================================

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        device = t.device
        freqs = torch.exp(-math.log(10000.0) * torch.arange(0, half, device=device).float() / max(half - 1, 1))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([args.sin(), args.cos()], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, time_dim: Optional[int] = None):
        super().__init__()
        self.time_dim = time_dim
        self.norm1 = nn.GroupNorm(num_groups(in_channels), in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups(out_channels), out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act = nn.SiLU()
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()
        self.time_proj = nn.Linear(time_dim, out_channels) if time_dim is not None else None

    def forward(self, x: torch.Tensor, t_emb: Optional[torch.Tensor] = None) -> torch.Tensor:
        h = self.conv1(self.act(self.norm1(x)))
        if self.time_proj is not None and t_emb is not None:
            h = h + self.time_proj(self.act(t_emb))[:, :, None, None]
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.skip(x)


class AttentionBlock(nn.Module):
    def __init__(self, channels: int, num_heads: int = 4):
        super().__init__()
        self.norm = nn.GroupNorm(num_groups(channels), channels)
        self.attn = nn.MultiheadAttention(channels, num_heads=num_heads, batch_first=True)
        self.proj = nn.Linear(channels, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        residual = x
        x = self.norm(x).reshape(b, c, h * w).permute(0, 2, 1)
        attn_out, _ = self.attn(x, x, x, need_weights=False)
        attn_out = self.proj(attn_out)
        attn_out = attn_out.permute(0, 2, 1).reshape(b, c, h, w)
        return residual + attn_out


class Downsample(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=4, stride=2, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)


# ============================================================
# Stage 1: ConvVAE (kept close to original)
# ============================================================

class ConvVAE(nn.Module):
    def __init__(self, in_channels: int = 3, base_channels: int = 64, latent_channels: int = 4):
        super().__init__()
        ch = base_channels

        self.enc_in = nn.Conv2d(in_channels, ch, kernel_size=3, padding=1)
        self.enc_block1 = ResidualBlock(ch, ch)
        self.down1 = Downsample(ch)
        self.enc_block2 = ResidualBlock(ch, ch * 2)
        self.down2 = Downsample(ch * 2)
        self.enc_block3 = ResidualBlock(ch * 2, ch * 4)
        self.enc_attn = AttentionBlock(ch * 4)

        self.to_mu = nn.Conv2d(ch * 4, latent_channels, kernel_size=1)
        self.to_logvar = nn.Conv2d(ch * 4, latent_channels, kernel_size=1)

        self.dec_in = nn.Conv2d(latent_channels, ch * 4, kernel_size=3, padding=1)
        self.dec_block1 = ResidualBlock(ch * 4, ch * 4)
        self.dec_attn = AttentionBlock(ch * 4)
        self.up1 = Upsample(ch * 4)
        self.dec_block2 = ResidualBlock(ch * 4, ch * 2)
        self.up2 = Upsample(ch * 2)
        self.dec_block3 = ResidualBlock(ch * 2, ch)
        self.out_norm = nn.GroupNorm(num_groups(ch), ch)
        self.out_conv = nn.Conv2d(ch, in_channels, kernel_size=3, padding=1)

    def encode_stats(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.enc_in(x)
        x = self.enc_block1(x)
        x = self.down1(x)
        x = self.enc_block2(x)
        x = self.down2(x)
        x = self.enc_block3(x)
        x = self.enc_attn(x)
        mu = self.to_mu(x)
        logvar = self.to_logvar(x).clamp(-10, 5)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std) * temperature
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        x = self.dec_in(z)
        x = self.dec_block1(x)
        x = self.dec_attn(x)
        x = self.up1(x)
        x = self.dec_block2(x)
        x = self.up2(x)
        x = self.dec_block3(x)
        x = self.out_conv(F.silu(self.out_norm(x)))
        return torch.tanh(x)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encode_stats(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


# ============================================================
# Stage 2: conditional latent diffusion U-Net
# ============================================================

class LatentUNetConditional(nn.Module):
    def __init__(self, latent_channels: int = 4, base_channels: int = 128, time_dim: int = 512, cond_dim: int = 18):
        super().__init__()
        self.cond_dim = cond_dim

        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.cond_embed = nn.Sequential(
            nn.Linear(cond_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        ch = base_channels
        self.in_conv = nn.Conv2d(latent_channels, ch, kernel_size=3, padding=1)

        self.down1a = ResidualBlock(ch, ch, time_dim=time_dim)
        self.down1b = ResidualBlock(ch, ch, time_dim=time_dim)
        self.attn1 = AttentionBlock(ch)
        self.downsample1 = Downsample(ch)

        self.down2a = ResidualBlock(ch, ch * 2, time_dim=time_dim)
        self.down2b = ResidualBlock(ch * 2, ch * 2, time_dim=time_dim)
        self.attn2 = AttentionBlock(ch * 2)
        self.downsample2 = Downsample(ch * 2)

        self.mid1 = ResidualBlock(ch * 2, ch * 4, time_dim=time_dim)
        self.mid_attn = AttentionBlock(ch * 4)
        self.mid2 = ResidualBlock(ch * 4, ch * 2, time_dim=time_dim)

        self.upsample2 = Upsample(ch * 2)
        self.up2a = ResidualBlock(ch * 4, ch * 2, time_dim=time_dim)
        self.up2b = ResidualBlock(ch * 2, ch, time_dim=time_dim)
        self.upattn2 = AttentionBlock(ch)

        self.upsample1 = Upsample(ch)
        self.up1a = ResidualBlock(ch * 2, ch, time_dim=time_dim)
        self.up1b = ResidualBlock(ch, ch, time_dim=time_dim)
        self.upattn1 = AttentionBlock(ch)

        self.out_norm = nn.GroupNorm(num_groups(ch), ch)
        self.out_conv = nn.Conv2d(ch, latent_channels, kernel_size=3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, x: torch.Tensor, t: torch.Tensor, cond: Optional[torch.Tensor] = None) -> torch.Tensor:
        if cond is None:
            cond = torch.zeros(x.size(0), self.cond_dim, device=x.device, dtype=x.dtype)
        t_emb = self.time_embed(t)
        c_emb = self.cond_embed(cond)
        emb = t_emb + c_emb

        x0 = self.in_conv(x)

        x1 = self.down1a(x0, emb)
        x1 = self.down1b(x1, emb)
        x1 = self.attn1(x1)
        x = self.downsample1(x1)

        x2 = self.down2a(x, emb)
        x2 = self.down2b(x2, emb)
        x2 = self.attn2(x2)
        x = self.downsample2(x2)

        x = self.mid1(x, emb)
        x = self.mid_attn(x)
        x = self.mid2(x, emb)

        x = self.upsample2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.up2a(x, emb)
        x = self.up2b(x, emb)
        x = self.upattn2(x)

        x = self.upsample1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.up1a(x, emb)
        x = self.up1b(x, emb)
        x = self.upattn1(x)

        return self.out_conv(F.silu(self.out_norm(x)))


# ============================================================
# Diffusion in latent space
# ============================================================

class LatentDiffusion:
    def __init__(
        self,
        timesteps: int = 300,
        objective: str = "v",
        min_snr_gamma: float = 5.0,
        device: str = "cuda",
    ):
        if objective not in {"v", "eps"}:
            raise ValueError("objective must be 'v' or 'eps'")
        self.timesteps = timesteps
        self.objective = objective
        self.min_snr_gamma = min_snr_gamma
        self.device = torch.device(device)

        betas = self.cosine_beta_schedule(timesteps).to(self.device)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = torch.cat([torch.ones(1, device=self.device), alphas_cumprod[:-1]], dim=0)

        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.alphas_cumprod_prev = alphas_cumprod_prev

        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
        self.posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

    @staticmethod
    def cosine_beta_schedule(timesteps: int, s: float = 0.008) -> torch.Tensor:
        steps = timesteps + 1
        x = torch.linspace(0, timesteps, steps)
        alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return betas.clamp(1e-4, 0.999)

    def extract(self, arr: torch.Tensor, t: torch.Tensor, shape: torch.Size) -> torch.Tensor:
        out = arr.gather(0, t)
        return out.view(t.shape[0], *([1] * (len(shape) - 1)))

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        alpha = self.extract(self.sqrt_alphas_cumprod, t, x0.shape)
        sigma = self.extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape)
        return alpha * x0 + sigma * noise

    def predict_v(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        alpha = self.extract(self.sqrt_alphas_cumprod, t, x0.shape)
        sigma = self.extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape)
        return alpha * noise - sigma * x0

    def predict_x0_from_v(self, xt: torch.Tensor, t: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        alpha = self.extract(self.sqrt_alphas_cumprod, t, xt.shape)
        sigma = self.extract(self.sqrt_one_minus_alphas_cumprod, t, xt.shape)
        return alpha * xt - sigma * v

    def predict_eps_from_v(self, xt: torch.Tensor, t: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        alpha = self.extract(self.sqrt_alphas_cumprod, t, xt.shape)
        sigma = self.extract(self.sqrt_one_minus_alphas_cumprod, t, xt.shape)
        return sigma * xt + alpha * v

    def model_predictions(self, model_out: torch.Tensor, xt: torch.Tensor, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.objective == "v":
            x0 = self.predict_x0_from_v(xt, t, model_out)
            eps = self.predict_eps_from_v(xt, t, model_out)
        else:
            eps = model_out
            alpha = self.extract(self.sqrt_alphas_cumprod, t, xt.shape)
            sigma = self.extract(self.sqrt_one_minus_alphas_cumprod, t, xt.shape)
            x0 = (xt - sigma * eps) / alpha.clamp(min=1e-8)
        return x0, eps

    def loss_weight(self, t: torch.Tensor) -> torch.Tensor:
        snr = self.alphas_cumprod[t] / (1.0 - self.alphas_cumprod[t]).clamp(min=1e-8)
        clipped = torch.minimum(snr, torch.full_like(snr, self.min_snr_gamma))
        if self.objective == "v":
            return clipped / (snr + 1.0)
        return clipped / snr.clamp(min=1e-8)

    @torch.no_grad()
    def ddim_sample(
        self,
        model: nn.Module,
        shape: Tuple[int, ...],
        steps: int = 50,
        eta: float = 0.0,
        temperature: float = 1.0,
        cond: Optional[torch.Tensor] = None,
        guidance_scale: float = 0.0,
    ) -> torch.Tensor:
        b = shape[0]
        x = torch.randn(shape, device=self.device) * temperature
        times = torch.linspace(self.timesteps - 1, 0, steps, device=self.device).long()
        next_times = torch.cat([times[1:], torch.tensor([-1], device=self.device, dtype=torch.long)], dim=0)

        for t_curr, t_next in zip(times, next_times):
            t = torch.full((b,), int(t_curr.item()), device=self.device, dtype=torch.long)

            if cond is None or guidance_scale == 0.0:
                model_out = model(x, t, cond)
            else:
                uncond = torch.zeros_like(cond)
                model_out_cond = model(x, t, cond)
                model_out_uncond = model(x, t, uncond)
                model_out = model_out_uncond + guidance_scale * (model_out_cond - model_out_uncond)

            x0, eps = self.model_predictions(model_out, x, t)
            x0 = x0.clamp(-4.0, 4.0)

            if t_next < 0:
                x = x0
                continue

            at = self.alphas_cumprod[t_curr]
            an = self.alphas_cumprod[t_next]
            sigma = eta * torch.sqrt(((1.0 - an) / (1.0 - at)).clamp(min=1e-8) * (1.0 - at / an).clamp(min=0.0))
            c = torch.sqrt((1.0 - an - sigma ** 2).clamp(min=0.0))
            noise = torch.randn_like(x) * temperature
            x = torch.sqrt(an) * x0 + c * eps + sigma * noise
        return x


# ============================================================
# Preview helpers
# ============================================================

@torch.no_grad()
def save_dataset_preview(loader: DataLoader, output_dir: str) -> None:
    images, _ = next(iter(loader))
    images = images[:16]
    save_grid(images, os.path.join(output_dir, "dataset_preview.png"), nrow=4)


@torch.no_grad()
def save_vae_reconstruction_grid(vae: ConvVAE, loader: DataLoader, device: torch.device, output_path: str) -> None:
    vae.eval()
    x, _ = next(iter(loader))
    x = x[:8].to(device)
    mu, _ = vae.encode_stats(x)
    recon = vae.decode(mu)
    grid = torch.cat([x, recon], dim=0)
    save_grid(grid, output_path, nrow=4)


@torch.no_grad()
def estimate_latent_scale(vae: ConvVAE, loader: DataLoader, device: torch.device, max_batches: int = 64) -> float:
    vae.eval()
    zs = []
    for i, (x, _) in enumerate(loader):
        x = x.to(device)
        mu, _ = vae.encode_stats(x)
        zs.append(mu.flatten())
        if i + 1 >= max_batches:
            break
    z = torch.cat(zs, dim=0)
    return float(1.0 / z.std().clamp(min=1e-6).item())


@torch.no_grad()
def sample_and_save_latent_diffusion(
    vae: ConvVAE,
    model: nn.Module,
    diffusion: LatentDiffusion,
    latent_scale: float,
    cfg: Config,
    output_dir: str,
    step: int,
    device: torch.device,
) -> None:
    vae.eval()
    model.eval()

    latent_h = cfg.image_size // 4
    latent_w = cfg.image_size // 4

    # --------------------------------------------------------
    # Unconditional sample
    # --------------------------------------------------------
    z_shape = (
        16,
        cfg.vae_latent_channels,
        latent_h,
        latent_w,
    )

    latents_uncond = diffusion.ddim_sample(
        model=model,
        shape=z_shape,
        steps=cfg.diff_ddim_steps,
        eta=cfg.sample_eta,
        temperature=cfg.sample_temperature,
        cond=None,
        guidance_scale=0.0,
    )

    images_uncond = vae.decode(latents_uncond / latent_scale)
    save_grid(
        images_uncond,
        os.path.join(output_dir, f"samples_uncond_step_{step:06d}.png"),
        nrow=4,
    )

    # --------------------------------------------------------
    # Type-conditional sample
    # --------------------------------------------------------
    cond = make_sample_conditions(cfg.sample_type_names, device=device)

    latents_types = diffusion.ddim_sample(
        model=model,
        shape=(
            cond.size(0),
            cfg.vae_latent_channels,
            latent_h,
            latent_w,
        ),
        steps=cfg.diff_ddim_steps,
        eta=cfg.sample_eta,
        temperature=cfg.sample_temperature,
        cond=cond,
        guidance_scale=cfg.guidance_scale,
    )

    images_types = vae.decode(latents_types / latent_scale)
    save_grid(
        images_types,
        os.path.join(output_dir, f"samples_types_step_{step:06d}.png"),
        nrow=4,
    )

    print(
        f"[SAMPLE] step={step:06d} "
        f"types={cfg.sample_type_names} guidance={cfg.guidance_scale:.2f}"
    )

# ============================================================
# Stage 1 training
# ============================================================

def train_vae(cfg: Config, train_loader: DataLoader, device: torch.device) -> Tuple[ConvVAE, float]:
    vae = ConvVAE(
        in_channels=3,
        base_channels=cfg.vae_base_channels,
        latent_channels=cfg.vae_latent_channels,
    ).to(device)

    optimizer = torch.optim.AdamW(vae.parameters(), lr=cfg.vae_lr, betas=(0.9, 0.99), weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.use_amp and device.type == "cuda"))

    stage_dir = os.path.join(cfg.output_dir, "stage1_vae")
    ensure_dir(stage_dir)

    global_step = 0
    for epoch in range(1, cfg.vae_epochs + 1):
        vae.train()
        running_loss = 0.0
        running_l1 = 0.0
        running_kl = 0.0
        count = 0

        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad(set_to_none=True)

            with device_autocast(device, cfg.use_amp):
                recon, mu, logvar = vae(x)
                l1 = F.l1_loss(recon, x)
                mse = F.mse_loss(recon, x)
                kl = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())
                kl_weight = cfg.vae_kl_weight * min(1.0, global_step / max(1, len(train_loader) * 20))
                loss = l1 + cfg.vae_recon_mse_weight * mse + kl_weight * kl

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(vae.parameters(), cfg.vae_grad_clip)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)
            running_l1 += l1.item() * x.size(0)
            running_kl += kl.item() * x.size(0)
            count += x.size(0)
            global_step += 1

        avg_loss = running_loss / max(count, 1)
        avg_l1 = running_l1 / max(count, 1)
        avg_kl = running_kl / max(count, 1)
        print(f"[VAE] epoch={epoch:04d} loss={avg_loss:.5f} l1={avg_l1:.5f} kl={avg_kl:.5f}")

        if epoch % cfg.vae_save_every == 0 or epoch == 1 or epoch == cfg.vae_epochs:
            save_vae_reconstruction_grid(
                vae,
                train_loader,
                device,
                os.path.join(stage_dir, f"recon_epoch_{epoch:04d}.png"),
            )
            torch.save(
                {
                    "model": vae.state_dict(),
                    "epoch": epoch,
                    "config": cfg.__dict__,
                },
                os.path.join(stage_dir, "vae_last.pt"),
            )

    latent_scale = estimate_latent_scale(vae, train_loader, device=device)
    print(f"[VAE] latent_scale={latent_scale:.6f}")
    torch.save(
        {
            "model": vae.state_dict(),
            "latent_scale": latent_scale,
            "config": cfg.__dict__,
        },
        os.path.join(stage_dir, "vae_final.pt"),
    )
    return vae, latent_scale


# ============================================================
# Stage 2 training
# ============================================================

class EMA:
    def __init__(self, beta: float):
        self.beta = beta

    @torch.no_grad()
    def update_model_average(self, ema_model: nn.Module, model: nn.Module) -> None:
        for current_params, ema_params in zip(model.parameters(), ema_model.parameters()):
            ema_params.data.mul_(self.beta).add_(current_params.data, alpha=1.0 - self.beta)

        for current_buffers, ema_buffers in zip(model.buffers(), ema_model.buffers()):
            ema_buffers.data.copy_(current_buffers.data)

def train_latent_diffusion(
    cfg: Config,
    train_loader: DataLoader,
    vae: ConvVAE,
    latent_scale: float,
    device: torch.device,
) -> None:
    vae.eval().requires_grad_(False)

    model = LatentUNetConditional(
        latent_channels=cfg.vae_latent_channels,
        base_channels=cfg.diff_base_channels,
        time_dim=cfg.diff_base_channels * 4,
        cond_dim=len(TYPE_LIST),
    ).to(device)

    ema_model = LatentUNetConditional(
        latent_channels=cfg.vae_latent_channels,
        base_channels=cfg.diff_base_channels,
        time_dim=cfg.diff_base_channels * 4,
        cond_dim=len(TYPE_LIST),
    ).to(device)
    ema_model.eval().requires_grad_(False)

    diffusion = LatentDiffusion(
        timesteps=cfg.diff_timesteps,
        objective=cfg.diff_objective,
        min_snr_gamma=cfg.diff_min_snr_gamma,
        device=str(device),
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.diff_lr,
        betas=(0.9, 0.99),
        weight_decay=1e-4,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.use_amp and device.type == "cuda"))
    ema = EMA(beta=cfg.diff_ema_decay)

    stage_dir = os.path.join(cfg.output_dir, "stage2_latent_diffusion")
    ensure_dir(stage_dir)

    start_step = 0

    # --------------------------------------------------------
    # Resume diffusion if checkpoint is provided
    # --------------------------------------------------------
    if cfg.diff_ckpt_path is not None:
        if not os.path.exists(cfg.diff_ckpt_path):
            raise FileNotFoundError(f"Diffusion checkpoint not found: {cfg.diff_ckpt_path}")

        ckpt = torch.load(cfg.diff_ckpt_path, map_location=device)

        model.load_state_dict(ckpt["model"], strict=True)

        if "ema_model" in ckpt and ckpt["ema_model"] is not None:
            ema_model.load_state_dict(ckpt["ema_model"], strict=True)
        else:
            ema_model.load_state_dict(model.state_dict())

        if "optimizer" in ckpt and ckpt["optimizer"] is not None:
            optimizer.load_state_dict(ckpt["optimizer"])

        if "scaler" in ckpt and ckpt["scaler"] is not None:
            scaler.load_state_dict(ckpt["scaler"])

        start_step = int(ckpt.get("step", 0))

        ckpt_latent_scale = ckpt.get("latent_scale", None)
        if ckpt_latent_scale is not None:
            latent_scale = float(ckpt_latent_scale)

        print(f"[RESUME] Loaded diffusion checkpoint: {cfg.diff_ckpt_path}")
        print(f"[RESUME] start_step={start_step}")
        print(f"[RESUME] latent_scale={latent_scale:.6f}")
    else:
        ema_model.load_state_dict(model.state_dict())

    batch_iter = cycle(train_loader)
    model.train()

    for step in range(start_step + 1, cfg.diff_train_steps + 1):
        x, cond = next(batch_iter)
        x = x.to(device, non_blocking=True)
        cond = cond.to(device, non_blocking=True)

        # ----------------------------------------------------
        # Latent encoding from pretrained probabilistic VAE
        # ----------------------------------------------------
        with torch.no_grad():
            mu, logvar = vae.encode_stats(x)
            if cfg.diff_use_posterior_sample:
                z = vae.reparameterize(
                    mu,
                    logvar,
                    temperature=cfg.diff_posterior_temperature,
                )
            else:
                z = mu
            z = z * latent_scale

        # ----------------------------------------------------
        # Classifier-free guidance dropout during training
        # ----------------------------------------------------
        if cfg.cond_drop_prob > 0.0:
            keep_mask = (torch.rand(cond.size(0), 1, device=device) > cfg.cond_drop_prob).float()
            cond_in = cond * keep_mask
        else:
            cond_in = cond

        # ----------------------------------------------------
        # Diffusion training
        # ----------------------------------------------------
        t = torch.randint(0, cfg.diff_timesteps, (x.size(0),), device=device)
        noise = torch.randn_like(z)
        zt = diffusion.q_sample(z, t, noise)

        target = diffusion.predict_v(z, t, noise) if cfg.diff_objective == "v" else noise
        weights = diffusion.loss_weight(t).view(-1, 1, 1, 1)

        optimizer.zero_grad(set_to_none=True)

        with device_autocast(device, cfg.use_amp):
            pred = model(zt, t, cond_in)
            loss = ((pred - target) ** 2 * weights).mean()

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.diff_grad_clip)
        scaler.step(optimizer)
        scaler.update()

        if step >= cfg.diff_ema_start:
            ema.update_model_average(ema_model, model)
        elif step == start_step + 1:
            ema_model.load_state_dict(model.state_dict())

        if step == start_step + 1 or step % 100 == 0:
            print(f"[DIFF] step={step:06d} loss={loss.item():.6f}")

        if step == start_step + 1 or step % cfg.diff_sample_every == 0:
            sample_and_save_latent_diffusion(
                vae=vae,
                model=ema_model if step >= cfg.diff_ema_start else model,
                diffusion=diffusion,
                latent_scale=latent_scale,
                cfg=cfg,
                output_dir=stage_dir,
                step=step,
                device=device,
            )

        if step % cfg.diff_save_every == 0 or step == cfg.diff_train_steps:
            torch.save(
                {
                    "model": model.state_dict(),
                    "ema_model": ema_model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(),
                    "latent_scale": latent_scale,
                    "step": step,
                    "config": cfg.__dict__,
                },
                os.path.join(stage_dir, "latent_diffusion_last.pt"),
            )





def load_vae_from_checkpoint(
    cfg: Config,
    ckpt_path: str,
    device: torch.device,
    train_loader: Optional[DataLoader] = None,
) -> Tuple[ConvVAE, float]:
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"VAE checkpoint not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device)
    ckpt_cfg = ckpt.get("config", {}) if isinstance(ckpt, dict) else {}

    ckpt_image_size = ckpt_cfg.get("image_size", None)
    if ckpt_image_size is not None and int(ckpt_image_size) != int(cfg.image_size):
        raise ValueError(
            f"Incompatible image_size between checkpoint ({ckpt_image_size}) and current CFG.image_size ({cfg.image_size})."
        )

    ckpt_base_channels = int(ckpt_cfg.get("vae_base_channels", cfg.vae_base_channels))
    ckpt_latent_channels = int(ckpt_cfg.get("vae_latent_channels", cfg.vae_latent_channels))

    vae = ConvVAE(
        in_channels=3,
        base_channels=ckpt_base_channels,
        latent_channels=ckpt_latent_channels,
    ).to(device)

    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    vae.load_state_dict(state, strict=True)
    vae.eval().requires_grad_(False)

    cfg.vae_base_channels = ckpt_base_channels
    cfg.vae_latent_channels = ckpt_latent_channels

    latent_scale = ckpt.get("latent_scale", None) if isinstance(ckpt, dict) else None
    if latent_scale is None:
        if train_loader is None:
            raise ValueError(
                "latent_scale absent du checkpoint et train_loader non fourni pour l'estimation."
            )
        print("[LOAD] latent_scale absent du checkpoint, estimation en cours...")
        latent_scale = estimate_latent_scale(vae, train_loader, device=device)
        print(f"[LOAD] latent_scale estimé={float(latent_scale):.6f}")

    print(f"[LOAD] VAE loaded from: {ckpt_path}")
    print(f"[LOAD] latent_scale={float(latent_scale):.6f}")
    print(f"[LOAD] vae_base_channels={cfg.vae_base_channels}")
    print(f"[LOAD] vae_latent_channels={cfg.vae_latent_channels}")

    return vae, float(latent_scale)

In [ ]:
# ============================================================
# Test guidance from saved checkpoints (single notebook cell)
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import torch
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
vae_ckpt_path = "/content/drive/MyDrive/outputs_pokemon_dataset_conditional_from_nv/stage1_vae/vae_last.pt"
diff_ckpt_path = "/content/drive/MyDrive/outputs_pokemon_dataset_conditional_4types_64/stage2_latent_diffusion/latent_diffusion_last.pt"

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
CFG.images_root = "/content/pokemon_data/Pokemon Dataset/Pokemon Dataset"
CFG.metadata_csv = "/content/pokemon_data/pokemonDB_dataset.csv"

CFG.image_size = 64
CFG.include_gif = False
CFG.shiny_mode = "both"
CFG.path_must_contain = ["normal"]
CFG.path_must_not_contain = ["back", "shiny", "back, shiny"]

CFG.batch_size = 128
CFG.num_workers = 4
CFG.use_amp = True

CFG.diff_ddim_steps = 100
CFG.sample_eta = 0.0
CFG.sample_temperature = 1.0

type_names = [
    "Fire", "Water", "Grass", "Electric",
    "Fire", "Water", "Grass", "Electric"
]

# ------------------------------------------------------------
# Runtime
# ------------------------------------------------------------
set_seed(CFG.seed)
device = torch.device(CFG.device)
print("Device:", device)

# ------------------------------------------------------------
# Dataset / loader
# ------------------------------------------------------------
dataset = PokemonDatasetConditional(
    images_root=CFG.images_root,
    metadata_csv=CFG.metadata_csv,
    image_size=CFG.image_size,
    include_gif=CFG.include_gif,
    shiny_mode=CFG.shiny_mode,
    path_must_contain=CFG.path_must_contain,
    path_must_not_contain=CFG.path_must_not_contain,
    limit_dataset=CFG.limit_dataset,
)

loader = DataLoader(
    dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(CFG.num_workers > 0),
    drop_last=True,
)

print(f"Dataset size: {len(dataset)}")

# ------------------------------------------------------------
# Load VAE
# ------------------------------------------------------------
vae_ckpt = torch.load(vae_ckpt_path, map_location=device)
vae_ckpt_cfg = vae_ckpt.get("config", {}) if isinstance(vae_ckpt, dict) else {}

if "image_size" in vae_ckpt_cfg:
    print("VAE checkpoint image_size:", vae_ckpt_cfg["image_size"])
    if vae_ckpt_cfg["image_size"] != CFG.image_size:
        raise ValueError(
            f"Incompatible VAE checkpoint: image_size={vae_ckpt_cfg['image_size']} "
            f"but CFG.image_size={CFG.image_size}. You need a 64x64 VAE here."
        )

if "vae_base_channels" in vae_ckpt_cfg:
    CFG.vae_base_channels = vae_ckpt_cfg["vae_base_channels"]
if "vae_latent_channels" in vae_ckpt_cfg:
    CFG.vae_latent_channels = vae_ckpt_cfg["vae_latent_channels"]

vae = ConvVAE(
    in_channels=3,
    base_channels=CFG.vae_base_channels,
    latent_channels=CFG.vae_latent_channels,
).to(device)

vae.load_state_dict(vae_ckpt["model"], strict=True)
vae.eval().requires_grad_(False)

latent_scale = vae_ckpt.get("latent_scale", None)
if latent_scale is None:
    print("latent_scale absent du VAE checkpoint, estimation en cours...")
    latent_scale = estimate_latent_scale(vae, loader, device=device)

print(f"Loaded VAE: {vae_ckpt_path}")
print(f"latent_scale = {latent_scale:.6f}")
print(f"vae_base_channels = {CFG.vae_base_channels}")
print(f"vae_latent_channels = {CFG.vae_latent_channels}")

# ------------------------------------------------------------
# Load diffusion checkpoint
# ------------------------------------------------------------
diff_ckpt = torch.load(diff_ckpt_path, map_location=device)

cond_dim = len(TYPE_LIST)

model = LatentUNetConditional(
    latent_channels=CFG.vae_latent_channels,
    base_channels=CFG.diff_base_channels,
    time_dim=CFG.diff_base_channels * 4,
    cond_dim=cond_dim,
).to(device)

ema_model = LatentUNetConditional(
    latent_channels=CFG.vae_latent_channels,
    base_channels=CFG.diff_base_channels,
    time_dim=CFG.diff_base_channels * 4,
    cond_dim=cond_dim,
).to(device)

model.load_state_dict(diff_ckpt["model"], strict=True)
ema_model.load_state_dict(diff_ckpt["ema_model"], strict=True)

model.eval()
ema_model.eval()

print(f"Loaded diffusion checkpoint: {diff_ckpt_path}")
print(f"Diffusion step: {diff_ckpt.get('step', 'unknown')}")

if "latent_scale" in diff_ckpt and diff_ckpt["latent_scale"] is not None:
    latent_scale = float(diff_ckpt["latent_scale"])
    print(f"latent_scale overridden from diffusion ckpt = {latent_scale:.6f}")

diffusion = LatentDiffusion(
    timesteps=CFG.diff_timesteps,
    objective=CFG.diff_objective,
    min_snr_gamma=CFG.diff_min_snr_gamma,
    device=str(device),
)

# ------------------------------------------------------------
# Compare guidance values
# ------------------------------------------------------------
@torch.no_grad()
def compare_guidance_scales(
    vae,
    model,
    diffusion,
    latent_scale,
    type_names,
    guidance_values=(0.0, 0.5, 1.0, 1.5, 2.0),
    seed=42,
    out_dir="/content/guidance_tests",
):
    os.makedirs(out_dir, exist_ok=True)

    device = next(model.parameters()).device
    latent_h = CFG.image_size // 4
    latent_w = CFG.image_size // 4

    cond = make_sample_conditions(type_names, device=device)

    for g in guidance_values:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        latents = diffusion.ddim_sample(
            model=model,
            shape=(len(type_names), CFG.vae_latent_channels, latent_h, latent_w),
            steps=CFG.diff_ddim_steps,
            eta=CFG.sample_eta,
            temperature=CFG.sample_temperature,
            cond=cond,
            guidance_scale=g,
        )

        images = vae.decode(latents / latent_scale)
        save_path = os.path.join(out_dir, f"guidance_{g:.1f}.png")
        save_grid(images, save_path, nrow=4)
        print(f"saved: {save_path}")

compare_guidance_scales(
    vae=vae,
    model=ema_model,
    diffusion=diffusion,
    latent_scale=latent_scale,
    type_names=type_names,
    guidance_values=(0.0, 0.5, 1.0, 1.5, 2.0),
    seed=42,
    out_dir="/content/guidance_tests",
)

print("Done. Check /content/guidance_tests")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Dataset size: 2169
VAE checkpoint image_size: 64
latent_scale absent du VAE checkpoint, estimation en cours...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Loaded VAE: /content/drive/MyDrive/outputs_pokemon_dataset_conditional_from_nv/stage1_vae/vae_last.pt
latent_scale = 0.746554
vae_base_channels = 64
vae_latent_channels = 4
Loaded diffusion checkpoint: /content/drive/MyDrive/outputs_pokemon_dataset_conditional_4types_64/stage2_latent_diffusion/latent_diffusion_last.pt
Diffusion step: 500000
latent_scale overridden from diffusion ckpt = 0.747148
saved: /content/guidance_tests/guidance_0.0.png
saved: /content/guidance_tests/guidance_0.5.png
saved: /content/guidance_tests/guidance_1.0.png
saved: /content/guidance_tests/guidance_1.5.png
saved: /content/guidance_tests/guidance_2.0.png
Done. Check /content/guidance_tests
